# FindToFill: Showcase Unit Testing

Notebook ini menjalankan **semua unit test otomatis** FindToFill dan merangkum hasilnya untuk demo.

| | |
|---|---|
| **Aplikasi** | FindToFill: pencari dispenser air dan charging station di stasiun KRL |
| **Tools test** | Jest + React Testing Library (TypeScript) |
| **Database saat test** | Supabase **palsu** (in-memory), jadi test tidak butuh internet dan tidak mengubah data asli |
| **Pemetaan test case** | TC-01 sampai TC-17 mengikuti `docs/BUILD_PLAN.md` |

**Cara menjalankan (VS Code):** klik **Select Kernel** lalu pilih Python 3. Kalau diminta memasang `ipykernel`, pilih **Install**. Setelah itu klik **Run All**. Jalankan dari folder `notebooks/` di dalam repo FindToFill (setelah `npm install`).

## 0. Persiapan

In [1]:
import json, os, re, subprocess, tempfile, time
from pathlib import Path

try:
    from IPython.display import Markdown, display
    def show(text): display(Markdown(text))
except ImportError:  # plain Python fallback
    def show(text): print(text)

# Repo root = the folder that holds package.json (this notebook lives in notebooks/).
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "package.json").exists())
TMP = Path(tempfile.mkdtemp(prefix="findtofill-tests-"))
ENV = {**{k: v for k, v in os.environ.items() if k != "CI"}, "FORCE_COLOR": "0"}  # CI=true hides --verbose output

def run(cmd):
    """Runs a shell command in the repo and returns (exit code, output)."""
    p = subprocess.run(cmd, shell=True, cwd=REPO, env=ENV, capture_output=True, text=True, encoding="utf-8", errors="replace")
    return p.returncode, (p.stdout or "") + (p.stderr or "")

for tool in ["node --version", "npm --version"]:
    print(f"{tool.split()[0]:5} {run(tool)[1].strip()}")
print("repo  ", REPO.name)

node  v26.13.0


npm   11.20.0
repo   FindToFill


## 1. Jalankan semua unit test

Satu perintah menjalankan seluruh test sekaligus mengukur coverage:
`npx jest --json --coverage`

In [2]:
started = time.time()
exit_code, log = run(
    f'npx jest --json --outputFile="{TMP / "jest.json"}" '
    f'--coverage --coverageReporters=json-summary --coverageDirectory="{TMP / "coverage"}"'
)
elapsed = time.time() - started
report = json.loads((TMP / "jest.json").read_text(encoding="utf-8"))

show(f"""
| Ringkasan | Hasil |
|---|---|
| File test (suite) | **{report["numPassedTestSuites"]} / {report["numTotalTestSuites"]} lulus** |
| Test case | **{report["numPassedTests"]} / {report["numTotalTests"]} lulus**, {report["numFailedTests"]} gagal |
| Waktu | {elapsed:.1f} detik |
| Status | {"**SEMUA LULUS**" if report["success"] else "**ADA YANG GAGAL** (lihat bagian 3)"} |
""")


| Ringkasan | Hasil |
|---|---|
| File test (suite) | **30 / 30 lulus** |
| Test case | **275 / 275 lulus**, 0 gagal |
| Waktu | 49.7 detik |
| Status | **SEMUA LULUS** |


## 2. Hasil per test case dan modul

Setiap file test dipetakan ke test case (TC) di `docs/BUILD_PLAN.md` dan ke bagian tim yang memilikinya.

In [3]:
# file test -> (test case, modul, bagian tim)
TC_MAP = {
    "migrations": ("TC-01", "Skema database dan RLS", "Backend"),
    "seed": ("TC-01", "Validasi data seed (CSV)", "Backend"),
    "apiErrors": ("TC-01", "Format error API", "Backend"),
    "roles": ("TC-02", "Login, role commuter/admin", "Backend"),
    "geo": ("TC-03", "Jarak haversine dan waktu jalan", "Backend"),
    "locator": ("TC-03", "Urutan fasilitas terdekat dan filter", "Backend"),
    "StationHeader": ("TC-03", "Header stasiun dan pilih stasiun", "Frontend"),
    "krlLines": ("TC-03", "Garis rute KRL di peta", "Frontend"),
    "navigation": ("TC-03", "Navigasi bawah", "Frontend"),
    "stationRoutes": ("TC-04", "API stasiun dan fasilitas", "Backend"),
    "stationService": ("TC-04", "Service stasiun (browser)", "Backend"),
    "mappers": ("TC-04", "Mapper data database ke aplikasi", "Backend"),
    "StatusBadge": ("TC-04", "Badge status (warna + teks + ikon)", "Frontend"),
    "submitReport": ("TC-05", "Kirim laporan dan stamp pertama", "Backend"),
    "reportService": ("TC-05", "Service laporan (browser)", "Backend"),
    "reportsRoute": ("TC-06", "API laporan: foto wajib, maks 5 MB", "Backend"),
    "reportInput": ("TC-06", "Validasi foto dan catatan", "Backend"),
    "rateLimit": ("TC-07", "Rate limit 5 laporan per jam", "Backend"),
    "status": ("TC-17", "Aturan status 7 hari", "Backend"),
    "submitReview": ("TC-08", "Ulasan: buat, edit, embedding", "Backend"),
    "reviewsRoute": ("TC-08", "API ulasan dan validasi", "Backend"),
    "embeddings": ("TC-08", "Embedding Gemini (768 dimensi)", "Backend"),
    "ReviewSheet": ("TC-08", "Form ulasan (Sudah diulas, edit)", "Frontend"),
    "favoritesStamps": ("TC-09", "Favorit dan stamp", "Backend"),
    "communityClient": ("TC-09", "Service komunitas (browser)", "Backend"),
    "schedules": ("TC-10-14", "Jadwal kereta berikutnya (WIB)", "Backend"),
    "reviewSearch": ("TC-10-14", "Pencarian ulasan untuk chatbot", "Backend"),
    "chatSupport": ("TC-10-14", "Rate limit dan log chat (privasi)", "Backend"),
    "groqClient": ("TC-10-14", "Klien Groq", "Backend"),
    "adminRoutes": ("TC-15", "Edit stasiun/fasilitas khusus admin", "Backend"),
    "adminManageRoutes": ("TC-15", "API admin: moderasi dan kelola data", "Backend"),
    "moderation": ("TC-16", "Remove (soft delete) laporan dan ulasan", "Backend"),
    "adminClient": ("TC-16", "Halaman admin: filter dan Remove", "Backend"),
}

def key_of(path):
    rel = Path(path).relative_to(REPO / "tests").as_posix()
    if rel.startswith("chatbot/"):
        return rel, ("TC-10-14", "Chatbot: " + Path(rel).stem.replace(".test", ""), "AI")
    stem = Path(rel).name.split(".")[0]
    return rel, TC_MAP.get(stem, ("-", stem, "-"))

rows = []
for suite in report["testResults"]:
    rel, (tc, module, owner) = key_of(suite["name"])
    tests = suite["assertionResults"]
    passed = sum(t["status"] == "passed" for t in tests)
    rows.append((tc, module, owner, rel, len(tests), passed))

rows.sort(key=lambda r: (r[0], r[3]))
table = "| TC | Modul | Bagian | File test | Test | Lulus | Status |\n|---|---|---|---|---|---|---|\n"
table += "\n".join(
    f"| {tc} | {module} | {owner} | `{rel}` | {n} | {ok} | {'LULUS' if ok == n else '**GAGAL**'} |"
    for tc, module, owner, rel, n, ok in rows
)
show(table)

by_owner = {}
for _, _, owner, _, n, ok in rows:
    total, good = by_owner.get(owner, (0, 0))
    by_owner[owner] = (total + n, good + ok)
show("**Per bagian tim:** " + " Â· ".join(f"{o}: {g}/{t} lulus" for o, (t, g) in sorted(by_owner.items())))

| TC | Modul | Bagian | File test | Test | Lulus | Status |
|---|---|---|---|---|---|---|
| TC-01 | Format error API | Backend | `apiErrors.test.ts` | 9 | 9 | LULUS |
| TC-01 | Skema database dan RLS | Backend | `migrations.test.ts` | 19 | 19 | LULUS |
| TC-01 | Validasi data seed (CSV) | Backend | `seed.test.ts` | 10 | 10 | LULUS |
| TC-02 | Login, role commuter/admin | Backend | `roles.test.ts` | 4 | 4 | LULUS |
| TC-03 | Jarak haversine dan waktu jalan | Backend | `geo.test.ts` | 8 | 8 | LULUS |
| TC-03 | Urutan fasilitas terdekat dan filter | Backend | `locator.test.ts` | 10 | 10 | LULUS |
| TC-04 | Badge status (warna + teks + ikon) | Frontend | `StatusBadge.test.tsx` | 5 | 5 | LULUS |
| TC-04 | Mapper data database ke aplikasi | Backend | `mappers.test.ts` | 5 | 5 | LULUS |
| TC-04 | API stasiun dan fasilitas | Backend | `stationRoutes.test.ts` | 11 | 11 | LULUS |
| TC-04 | Service stasiun (browser) | Backend | `stationService.test.ts` | 4 | 4 | LULUS |
| TC-05 | Service laporan (browser) | Backend | `reportService.test.ts` | 3 | 3 | LULUS |
| TC-05 | Kirim laporan dan stamp pertama | Backend | `submitReport.test.ts` | 8 | 8 | LULUS |
| TC-06 | Validasi foto dan catatan | Backend | `reportInput.test.ts` | 16 | 16 | LULUS |
| TC-06 | API laporan: foto wajib, maks 5 MB | Backend | `reportsRoute.test.ts` | 11 | 11 | LULUS |
| TC-07 | Rate limit 5 laporan per jam | Backend | `rateLimit.test.ts` | 4 | 4 | LULUS |
| TC-08 | Form ulasan (Sudah diulas, edit) | Frontend | `ReviewSheet.test.tsx` | 5 | 5 | LULUS |
| TC-08 | Embedding Gemini (768 dimensi) | Backend | `embeddings.test.ts` | 8 | 8 | LULUS |
| TC-08 | API ulasan dan validasi | Backend | `reviewsRoute.test.ts` | 14 | 14 | LULUS |
| TC-08 | Ulasan: buat, edit, embedding | Backend | `submitReview.test.ts` | 11 | 11 | LULUS |
| TC-09 | Service komunitas (browser) | Backend | `communityClient.test.ts` | 8 | 8 | LULUS |
| TC-09 | Favorit dan stamp | Backend | `favoritesStamps.test.ts` | 8 | 8 | LULUS |
| TC-10-14 | Rate limit dan log chat (privasi) | Backend | `chatSupport.test.ts` | 9 | 9 | LULUS |
| TC-10-14 | Klien Groq | Backend | `groqClient.test.ts` | 3 | 3 | LULUS |
| TC-10-14 | Pencarian ulasan untuk chatbot | Backend | `reviewSearch.test.ts` | 5 | 5 | LULUS |
| TC-10-14 | Jadwal kereta berikutnya (WIB) | Backend | `schedules.test.ts` | 14 | 14 | LULUS |
| TC-15 | API admin: moderasi dan kelola data | Backend | `adminManageRoutes.test.ts` | 25 | 25 | LULUS |
| TC-15 | Edit stasiun/fasilitas khusus admin | Backend | `adminRoutes.test.ts` | 6 | 6 | LULUS |
| TC-16 | Halaman admin: filter dan Remove | Backend | `adminClient.test.tsx` | 10 | 10 | LULUS |
| TC-16 | Remove (soft delete) laporan dan ulasan | Backend | `moderation.test.ts` | 8 | 8 | LULUS |
| TC-17 | Aturan status 7 hari | Backend | `status.test.ts` | 14 | 14 | LULUS |

**Per bagian tim:** Backend: 265/265 lulus Â· Frontend: 10/10 lulus

## 3. Aturan bisnis yang dibuktikan oleh test

Contoh nama test dari modul inti. Setiap baris adalah satu skenario yang dijalankan otomatis.

In [4]:
SHOWCASE = {
    "submitReport": "Laporan kondisi (foto wajib, rate limit, stamp)",
    "reportsRoute": "Validasi upload foto",
    "status": "Status fasilitas: hanya laporan 7 hari terakhir",
    "submitReview": "Ulasan dan embedding",
    "schedules": "Jadwal kereta dan cek 'sempat isi air'",
    "chatSupport": "Privasi: IP di-hash, log tanpa data pribadi",
    "moderation": "Admin Remove (soft delete)",
}

for suite in report["testResults"]:
    rel, _ = key_of(suite["name"])
    stem = Path(rel).name.split(".")[0]
    if stem not in SHOWCASE:
        continue
    lines = [f"**{SHOWCASE[stem]}** (`{rel}`)", ""]
    for t in suite["assertionResults"]:
        title = " > ".join([*t["ancestorTitles"][1:], t["title"]]).replace("|", "/")
        mark = "LULUS" if t["status"] == "passed" else "**GAGAL**"
        lines.append(f"- [{mark}] {title}")
    show("\n".join(lines))

failed = [
    (key_of(s["name"])[0], t["fullName"], (t.get("failureMessages") or [""])[0][:300])
    for s in report["testResults"] for t in s["assertionResults"] if t["status"] == "failed"
]
show("**Test yang gagal:** tidak ada." if not failed else "\n".join(f"- `{f}`: {n}\n\n```\n{m}\n```" for f, n, m in failed))

**Privasi: IP di-hash, log tanpa data pribadi** (`chatSupport.test.ts`)

- [LULUS] takes the first x-forwarded-for address, then x-real-ip
- [LULUS] hashes the IP with the secret salt (stable, not reversible without it)
- [LULUS] refuses to hash without IP_HASH_SALT in production
- [LULUS] uses the user id when logged in and never the raw IP for guests
- [LULUS] counts the request after the check passes
- [LULUS] rejects the 21st request with 429 and records nothing
- [LULUS] keeps a valid anonymous session id and replaces anything else
- [LULUS] writes only the allowed columns, with clean tool names and a rounded latency
- [LULUS] never throws, even when the insert fails

**Jadwal kereta dan cek 'sempat isi air'** (`schedules.test.ts`)

- [LULUS] converts UTC to Jakarta time (UTC+7)
- [LULUS] uses the weekend timetable on Saturday and Sunday in Jakarta, even when it is still Friday in UTC
- [LULUS] switches back to weekday after Sunday midnight WIB
- [LULUS] returns the next 2 per line and direction, soonest first, skipping trains already gone
- [LULUS] counts a train leaving this minute
- [LULUS] is empty after the last train of the day
- [LULUS] walk 3 min, train in 8 min
- [LULUS] walk 3 min, train in 7 min
- [LULUS] walk 4 min, train in 8 min
- [LULUS] reads only this station and today’s day type, and adds a time check
- [LULUS] filters by line when given
- [LULUS] reports no data (never a guess) when the station has no timetable
- [LULUS] reports service ended when every train today has left
- [LULUS] ignores an invalid walk time

**Admin Remove (soft delete)** (`moderation.test.ts`)

- [LULUS] lists reports (removed included) with facility, station, reporter, and a signed thumbnail
- [LULUS] filters by station (through its facilities) and status
- [LULUS] returns [] for a station without facilities without querying reports
- [LULUS] lists reviews with names and without the embedding
- [LULUS] sets removed_at and removed_by once, on a row that is not removed yet
- [LULUS] is safe to repeat: an already removed item returns its original removal
- [LULUS] returns 404 for an id that does not exist
- [LULUS] returns 403 when RLS silently refuses the update (not an admin in the database)

**Validasi upload foto** (`reportsRoute.test.ts`)

- [LULUS] rejects guests with 401 and saves nothing
- [LULUS] requires a photo
- [LULUS] rejects a non-image renamed to .jpg with 415 (content is checked, not the name)
- [LULUS] rejects photos over 5 MB with 413
- [LULUS] returns 400 for an unknown status
- [LULUS] returns 400 for a note over 140 characters
- [LULUS] returns 400 for a facility id that is not a UUID
- [LULUS] saves a valid report and returns 201 with the stamp flag
- [LULUS] passes the rate-limit error through as 429
- [LULUS] returns 401 for guests
- [LULUS] returns the signed-in user's reports

**Ulasan dan embedding** (`submitReview.test.ts`)

- [LULUS] creates the first review with its embedding and awards the stamp
- [LULUS] edits the existing review with an update (not an upsert) and awards no stamp
- [LULUS] looks up only the signed-in user’s review for this facility
- [LULUS] still saves the review with a null embedding when embedding fails
- [LULUS] clears the old vector when an edit cannot be embedded
- [LULUS] returns 404 for a missing or inactive facility before embedding
- [LULUS] returns 429 when the hourly review limit is reached, before embedding
- [LULUS] returns 409 when the user cannot review again (moderated review hidden by RLS)
- [LULUS] still succeeds when the stamp step fails after the review is saved
- [LULUS] returns non-removed reviews of the station facilities with facility names
- [LULUS] skips the reviews query for a station without facilities

**Laporan kondisi (foto wajib, rate limit, stamp)** (`submitReport.test.ts`)

- [LULUS] uploads the photo to the user folder, saves the report, and awards the first stamp
- [LULUS] does not award a second stamp for the same facility
- [LULUS] stores an empty note as null
- [LULUS] rejects the 6th report within the hour with 429 before uploading anything
- [LULUS] returns 404 for a missing or inactive facility before uploading
- [LULUS] returns 503 and saves nothing when the photo upload fails
- [LULUS] deletes the uploaded photo when the report insert fails
- [LULUS] still succeeds when the stamp step fails after the report is saved

**Status fasilitas: hanya laporan 7 hari terakhir** (`status.test.ts`)

- [LULUS] isReportActive > returns true for reports submitted 1 day ago
- [LULUS] isReportActive > returns true for reports submitted 6 days and 23 hours ago
- [LULUS] isReportActive > returns false for reports older than 7 days
- [LULUS] isReportActive > returns false for reports that have been removed by an admin
- [LULUS] resolveFacilityStatus > returns latest report within 7-day window
- [LULUS] resolveFacilityStatus > returns "unknown" (No recent report) when reports are older than 7 days
- [LULUS] getStatusBadgeInfo > maps "available" to Available with green styling
- [LULUS] getStatusBadgeInfo > maps "empty" to Empty with amber styling
- [LULUS] getStatusBadgeInfo > maps "broken" to Broken with red styling
- [LULUS] getStatusBadgeInfo > maps "under_maintenance" to Under Maintenance with slate styling
- [LULUS] getStatusBadgeInfo > maps "unknown" to No recent report with neutral gray styling
- [LULUS] formatTimeAgo > returns "No recent report" for null date
- [LULUS] formatTimeAgo > returns "Just now" for dates within 1 minute
- [LULUS] formatTimeAgo > returns minutes ago

**Test yang gagal:** tidak ada.

## 4. Coverage

Persentase kode yang dijalankan oleh test. **Catatan:** Jest hanya menghitung file yang di-import oleh test, jadi halaman yang belum punya test tidak ikut dihitung. Angka ini bukan coverage seluruh aplikasi.

In [5]:
summary = json.loads((TMP / "coverage" / "coverage-summary.json").read_text(encoding="utf-8"))
total = summary.pop("total")
show("| Metrik | Coverage |\n|---|---|\n" + "\n".join(
    f"| {label} | **{total[key]['pct']:.1f}%** ({total[key]['covered']}/{total[key]['total']}) |"
    for key, label in [("lines", "Baris (lines)"), ("statements", "Pernyataan (statements)"),
                       ("branches", "Cabang if/else (branches)"), ("functions", "Fungsi (functions)")]
))

AREAS = [
    ("src/app/api", "API routes"), ("src/lib/data", "Akses data (query database)"),
    ("src/lib/api", "Validasi input (zod)"), ("src/lib/services", "Service browser"),
    ("src/lib/chatbot", "Chatbot"), ("src/components", "Komponen UI"), ("src/lib", "Library lain"),
]
grouped = {}
for path, metrics in summary.items():
    rel = Path(path).resolve().relative_to(REPO).as_posix()
    area = next((label for prefix, label in AREAS if rel.startswith(prefix)), "Lainnya")
    lines = grouped.setdefault(area, [0, 0])
    lines[0] += metrics["lines"]["covered"]
    lines[1] += metrics["lines"]["total"]

show("| Area | Coverage baris |\n|---|---|\n" + "\n".join(
    f"| {area} | {100 * c / t:.1f}% ({c}/{t}) |" for area, (c, t) in grouped.items() if t
))

| Metrik | Coverage |
|---|---|
| Baris (lines) | **89.1%** (3708/4163) |
| Pernyataan (statements) | **89.1%** (3708/4163) |
| Cabang if/else (branches) | **83.9%** (630/751) |
| Fungsi (functions) | **82.8%** (178/215) |

| Area | Coverage baris |
|---|---|
| API routes | 97.2% (491/505) |
| Komponen UI | 75.5% (648/858) |
| Lainnya | 95.7% (179/187) |
| Library lain | 92.2% (887/962) |
| Validasi input (zod) | 99.7% (337/338) |
| Akses data (query database) | 91.0% (870/956) |
| Service browser | 82.9% (296/357) |

## 5. Demo langsung: satu file test

Menjalankan test laporan kondisi dengan nama setiap skenario (`--verbose`), sama seperti di terminal.

In [6]:
code_, out = run("npx jest tests/submitReport.test.ts --verbose --reporters=default")
print(re.sub(r"\x1b\[[0-9;]*m", "", out).strip())

PASS tests/submitReport.test.ts
  submitReport
    √ uploads the photo to the user folder, saves the report, and awards the first stamp (9 ms)
    √ does not award a second stamp for the same facility
    √ stores an empty note as null
    √ rejects the 6th report within the hour with 429 before uploading anything (1 ms)
    √ returns 404 for a missing or inactive facility before uploading (1 ms)
    √ returns 503 and saves nothing when the photo upload fails (1 ms)
    √ deletes the uploaded photo when the report insert fails (1 ms)
    √ still succeeds when the stamp step fails after the report is saved (3 ms)

Test Suites: 1 passed, 1 total
Tests:       8 passed, 8 total
Snapshots:   0 total
Time:        1.765 s, estimated 9 s
Ran all test suites matching tests/submitReport.test.ts.


## 6. Kesimpulan

- Semua unit test dijalankan otomatis dengan satu perintah (`npm test`), tanpa internet dan tanpa menyentuh database asli.
- Yang sudah teruji: keamanan database dan role, pencari fasilitas terdekat, laporan dengan foto dan rate limit, aturan status 7 hari, ulasan dengan embedding, favorit, stamp, helper chatbot (jadwal, pencarian ulasan, privasi log), dan panel admin.
- **Batasan:** orchestrator chatbot (Groq dan 5 tool) masih dikembangkan, jadi evaluasinya (20 kasus) menyusul. Alur di browser seperti login dan upload foto dicek secara manual.